In [0]:
#auto loader
df=spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format","csv")\
    .option("cloudFiles.schemaLocation","/Volumes/workspace/default/autoloader_sink/schema1") \
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
    .option("cloudFiles.cleanSource", "MOVE") \
    .option("cloudFiles.cleanSource.moveDestination",
        "/Volumes/workspace/default/autoloader_sink/archive") \
    .load("/Volumes/workspace/default/autoloader_source/source")

In [0]:
df.writeStream.format("delta") \
    .option("checkPointLocation","/Volumes/workspace/default/autoloader_sink/checkpoint1") \
    .option("mergeSchema","True") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable("autoloader_sink")
    

In [0]:
#delete directory
dbutils.fs.rm("/Volumes/workspace/default/autoloader_sink/checkpoint1/",recurse=True)
dbutils.fs.rm("/Volumes/workspace/default/autoloader_sink/schema1/",recurse=True)
spark.sql("DROP TABLE IF EXISTS autoloader_sink")


In [0]:
%sql
select * from autoloader_sink

### Schema Evolution without autoloader

In [0]:
data1=[(1,"USA","New York"),
       (2,"USA","California"),
       (3,"Canada","Toronto"),
       (4,"England","London"),
       (5,"Mexico","Mexico City")]
data2=[(1,"USA","New York","NY"),
       (2,"USA","California","CA"),
       (3,"Canada","Toronto","ON"),
       (4,"England","London","scotland")]
data3=[(1,"USA","New York"),
       (2,"USA","California")]

#create data frames
df1=spark.createDataFrame(data1,['customer_id','country','city'])
df2=spark.createDataFrame(data2,['customer_id','country','city','state'])
df3=spark.createDataFrame(data3,['customer_id','country','city'])

#write data frame 1 to delta table df2
spark.sql(""" drop table if exists df2""")
df1.write.format("delta").mode("append").saveAsTable("df2")

# write data frame 2 into table df2
df2.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable("df2")

#write data frame 3 into table df2
df3.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable("df2")

In [0]:
%sql
select * from df2

## Flatten the List

In [0]:
from pyspark.sql.functions import *
data1=[("Ravi","Cricket,badminton"),
       ("Siddu","Swimming,Cricket"),
       ("Aarush","Foot Ball,Cricket")
]

df=spark.createDataFrame(data1,["name","sport"])
df.select("name",explode(split("sport",","))).orderBy(col("name")).display()

In [0]:
from pyspark.sql.functions import *
data1=["cricket,swimming"]
df=spark.createDataFrame(data1,["sport"])
df.display()
df=df.withColumn("sport",split("sport",","))
df.display()
df.select(explode("sport")).display()
